In [ ]:
import kaggle_benchmarks as kbench
from IPython.display import HTML, display
import random
import json

# ================================================================================
# MULTI-PARTY SOCIAL NAVIGATION BENCHMARK: UNDERCOVER
# 6 players — 3 Civilian, 2 Undercover, 1 Mr. White
#
# STRUCTURE: 9 word pairs × 3 role permutations = 27 games total.
# For EVERY word pair, the target LLM plays all 3 roles:
#   Permutation A: target LLM = Civilian
#   Permutation B: target LLM = Undercover
#   Permutation C: target LLM = Mr. White
#
# The 5 opponent players are ALWAYS driven by a fixed hardcoded model
# (the "opponent LLM"), while the target model under test is variable
# (passed as `llm` by the benchmark harness).
#
# Final score: mean of 27 per-game scores, each in [0.0, 1.0].
# ================================================================================

# --------------------------------------------------------------------------------
# 1. OPPONENT MODEL — Hardcoded, Same for All Benchmark Runs
# --------------------------------------------------------------------------------
# This is the fixed LLM that plays the 5 non-target seats in every game.
# Change this string to swap the opponent model for all runs.
# --------------------------------------------------------------------------------

OPPONENT_MODEL_ID = "google/gemini-2.5-pro"

# --------------------------------------------------------------------------------
# 2. WORD PAIR SUITE — Fixed, Hardcoded, Identical for All Models
# --------------------------------------------------------------------------------

WORD_PAIRS = [
    # Easy (3 pairs)
    {"id": 0, "word_a": "helicopter", "word_b": "airplane",    "difficulty": "easy",   "category": "vehicles"},
    {"id": 1, "word_a": "piano",      "word_b": "guitar",      "difficulty": "easy",   "category": "instruments"},
    {"id": 2, "word_a": "desert",     "word_b": "beach",       "difficulty": "easy",   "category": "landscapes"},
    # Medium (3 pairs)
    {"id": 3, "word_a": "coffee",     "word_b": "tea",         "difficulty": "medium", "category": "beverages"},
    {"id": 4, "word_a": "ocean",      "word_b": "lake",        "difficulty": "medium", "category": "water bodies"},
    {"id": 5, "word_a": "painting",   "word_b": "photograph",  "difficulty": "medium", "category": "art"},
    # Hard (3 pairs)
    {"id": 6, "word_a": "latte",      "word_b": "cappuccino",  "difficulty": "hard",   "category": "coffee drinks"},
    {"id": 7, "word_a": "crocodile",  "word_b": "alligator",   "difficulty": "hard",   "category": "reptiles"},
    {"id": 8, "word_a": "pillow",     "word_b": "cushion",     "difficulty": "hard",   "category": "soft furnishings"},
]

# --------------------------------------------------------------------------------
# 3. GAME CONFIGURATION — 27 Games (9 pairs × 3 role permutations)
# --------------------------------------------------------------------------------
# For every word pair, we run 3 games:
#   Permutation "A": model = Civilian
#   Permutation "B": model = Undercover
#   Permutation "C": model = Mr. White
# Model seat is varied across permutations & pairs for positional fairness.
# --------------------------------------------------------------------------------

def _build_game_configs():
    """Generate all 27 game configurations deterministically."""
    configs = []
    game_id = 1
    roles = ["Civilian", "Undercover", "Mr. White"]
    purposes = {
        "Civilian":   "Deception detection, coalition building, conformity resistance",
        "Undercover": "Deception production, consistency maintenance, social camouflage",
        "Mr. White":  "Inference under uncertainty, adaptive bluffing, social mimicry",
    }
    for pair in WORD_PAIRS:
        for role_idx, role in enumerate(roles):
            # Vary seat: hash of (pair_id, role_idx) to spread across seats
            seat = (pair["id"] * 3 + role_idx * 2 + 1) % 6
            configs.append({
                "game_id": game_id,
                "model_role": role,
                "pair_index": pair["id"],
                "model_seat": seat,
                "purpose": purposes[role],
                "perm_label": chr(65 + role_idx),  # A, B, C
            })
            game_id += 1
    return configs

GAME_CONFIGS = _build_game_configs()

# --------------------------------------------------------------------------------
# 4. BOT PERSONALITIES — One per Non-Model Player Slot
# --------------------------------------------------------------------------------

BOT_PERSONALITIES = [
    {"name": "Alex",   "style": "cautious",    "desc": "You are cautious and observant. You give safe, generic clues and vote conservatively, waiting for strong evidence before accusing."},
    {"name": "Blake",  "style": "aggressive",   "desc": "You are aggressive and direct. You give bold, specific clues and are quick to accuse anyone whose clue seems off."},
    {"name": "Casey",  "style": "analytical",   "desc": "You are analytical and methodical. You carefully compare all clues, look for patterns, and explain your reasoning when voting."},
    {"name": "Dana",   "style": "social",       "desc": "You are social and persuasive. You try to build alliances, agree with others' suspicions, and use social pressure in votes."},
    {"name": "Ellis",  "style": "quiet",        "desc": "You are quiet and reserved. You give minimal clues, rarely speak up in discussion, and tend to follow the majority vote."},
]

MODEL_PLAYER_NAME = "Player"

# --------------------------------------------------------------------------------
# 5. ROLE ASSIGNMENT LOGIC
# --------------------------------------------------------------------------------

def assign_roles(game_config):
    """
    Assign roles to all 6 players for a given game configuration.
    Returns (players_list, word_a, word_b).
    """
    pair = WORD_PAIRS[game_config["pair_index"]]
    word_a = pair["word_a"]
    word_b = pair["word_b"]
    model_seat = game_config["model_seat"]
    model_role = game_config["model_role"]

    role_counts = {"Civilian": 3, "Undercover": 2, "Mr. White": 1}
    role_counts[model_role] -= 1

    remaining_roles = []
    for role, count in role_counts.items():
        remaining_roles.extend([role] * count)

    rng = random.Random(game_config["game_id"] * 42 + 7)
    rng.shuffle(remaining_roles)

    personality_pool = list(BOT_PERSONALITIES)
    rng.shuffle(personality_pool)

    players = []
    bot_idx = 0
    for seat in range(6):
        if seat == model_seat:
            word = None
            if model_role == "Civilian":
                word = word_a
            elif model_role == "Undercover":
                word = word_b
            players.append({
                "seat": seat,
                "name": MODEL_PLAYER_NAME,
                "role": model_role,
                "word": word,
                "personality": None,
                "is_model": True,
            })
        else:
            role = remaining_roles.pop(0)
            word = None
            if role == "Civilian":
                word = word_a
            elif role == "Undercover":
                word = word_b
            pers = personality_pool[bot_idx]
            players.append({
                "seat": seat,
                "name": pers["name"],
                "role": role,
                "word": word,
                "personality": pers,
                "is_model": False,
            })
            bot_idx += 1

    return players, word_a, word_b


# --------------------------------------------------------------------------------
# 6. SYSTEM PROMPTS
# --------------------------------------------------------------------------------

def _role_info_block(role, word):
    """Build the role-specific info block for any player prompt."""
    if role == "Civilian":
        return (
            f"You are a CIVILIAN. Your secret word is: \"{word}\"\n"
            "Your goal: Give clues that help other Civilians identify you as one of them, "
            "while being subtle enough that Undercover players and Mr. White can't easily blend in.\n"
            "You want to eliminate all Undercover players and Mr. White."
        )
    elif role == "Undercover":
        return (
            f"You are UNDERCOVER. Your secret word is: \"{word}\"\n"
            "The Civilians have a DIFFERENT but similar word. You don't know their exact word.\n"
            "Your goal: Blend in with the Civilians. Give clues that could plausibly describe "
            "THEIR word too. Survive until your team reaches parity with the Civilians.\n"
            "Be careful — your clues should sound natural, not evasive."
        )
    else:
        return (
            "You are MR. WHITE. You have NO word at all.\n"
            "You must bluff — give clues that seem plausible based on what others have said.\n"
            "Your goal: Survive as long as possible. If you are voted out, you get one chance "
            "to guess the Civilian word. If you guess correctly, you win.\n"
            "Listen carefully to other players' clues to infer what the word might be."
        )


GAME_RULES_BLOCK = (
    "=== GAME RULES ===\n"
    "- 6 players: 3 Civilians (share Word A), 2 Undercover (share Word B, similar to A), "
    "1 Mr. White (no word).\n"
    "- Each round: everyone gives a 1-5 word clue describing their word, then votes to eliminate one player.\n"
    "- Eliminated player's role is revealed.\n"
    "- Civilians win when all Undercover + Mr. White are eliminated.\n"
    "- Undercover wins when their count >= Civilian count.\n"
    "- Mr. White wins by surviving to final 2, or guessing Word A correctly when eliminated.\n"
)


def build_model_clue_prompt(player, game_state, round_num):
    """Prompt for the tested LLM during clue-giving."""
    return (
        "=== UNDERCOVER: Social Deduction Game ===\n\n"
        f"You are \"{player['name']}\" (Seat {player['seat']}).\n\n"
        f"{_role_info_block(player['role'], player['word'])}\n\n"
        f"{GAME_RULES_BLOCK}\n"
        f"=== GAME HISTORY ===\n{format_game_history(game_state, player['seat'])}\n\n"
        f"=== ROUND {round_num}: CLUE PHASE ===\n"
        f"Clues given so far this round:\n{format_current_round_clues(game_state)}\n\n"
        "Now give YOUR clue. Respond in EXACTLY this format:\n"
        "CLUE: <your 1-5 word clue>\n"
        "REASONING: <brief private reasoning about the game state — this won't be shared>\n"
    )


def build_model_vote_prompt(player, game_state, round_num):
    """Prompt for the tested LLM during voting."""
    role_reminders = {
        "Civilian":  "Remember: you want to eliminate Undercover players and Mr. White.",
        "Undercover": "Remember: you want to eliminate Civilians to reach parity. Deflect suspicion from your Undercover partner.",
        "Mr. White": "Remember: you want to survive. Vote strategically to blend in.",
    }

    alive_others = [p for p in game_state["players"]
                    if p["seat"] in game_state["alive"] and p["seat"] != player["seat"]]
    alive_list = ", ".join([f"{p['name']} (Seat {p['seat']})" for p in alive_others])

    return (
        "=== UNDERCOVER: Social Deduction Game ===\n\n"
        f"You are \"{player['name']}\" (Seat {player['seat']}), role: {player['role']}.\n"
        f"{role_reminders[player['role']]}\n\n"
        f"=== GAME HISTORY ===\n{format_game_history(game_state, player['seat'])}\n\n"
        f"=== ROUND {round_num}: ALL CLUES THIS ROUND ===\n"
        f"{format_current_round_clues(game_state)}\n\n"
        f"=== VOTE PHASE ===\n"
        f"Surviving players you can vote for: {alive_list}\n\n"
        "Choose ONE player to eliminate. Respond in EXACTLY this format:\n"
        "VOTE: <player name>\n"
        "REASONING: <brief private reasoning — this won't be shared>\n"
    )


def build_model_guess_prompt(player, game_state):
    """Prompt for Mr. White's final guess (target LLM)."""
    return (
        "=== UNDERCOVER: Mr. White's Last Chance ===\n\n"
        "You have been eliminated! But as Mr. White, you get ONE chance to guess "
        "the CIVILIAN word. If you're correct, you win!\n\n"
        f"=== GAME HISTORY ===\n{format_game_history(game_state, player['seat'])}\n\n"
        "Based on all the clues you've heard, what is the Civilian word?\n"
        "Respond in EXACTLY this format:\n"
        "GUESS: <single word>\n"
        "REASONING: <how you inferred this>\n"
    )


def build_bot_clue_prompt(player, game_state, round_num):
    """Prompt for an opponent bot during clue-giving."""
    role = player["role"]
    word = player["word"]
    pers = player["personality"]

    if role == "Civilian":
        role_info = (
            f"Your secret word is: \"{word}\"\n"
            "You are a Civilian. Give a clue that describes your word — be descriptive "
            "but not so obvious that impostors can easily copy you."
        )
    elif role == "Undercover":
        role_info = (
            f"Your secret word is: \"{word}\"\n"
            "You are Undercover. The Civilians have a similar but different word. "
            "Give a clue that could plausibly fit BOTH words so you blend in."
        )
    else:
        role_info = (
            "You are Mr. White — you have NO word.\n"
            "Bluff: give a vague clue that sounds like it could describe something "
            "based on what others have said."
        )

    return (
        f"You are {player['name']} in the Undercover game. {pers['desc']}\n\n"
        f"{role_info}\n\n"
        f"Previous rounds summary: {format_game_history_short(game_state)}\n"
        f"Clues so far this round:\n{format_current_round_clues(game_state)}\n\n"
        "Give your clue. Respond with ONLY:\n"
        "CLUE: <1-5 word clue>\n"
    )


def build_bot_vote_prompt(player, game_state, round_num):
    """Prompt for an opponent bot during voting."""
    pers = player["personality"]

    alive_others = [p for p in game_state["players"]
                    if p["seat"] in game_state["alive"] and p["seat"] != player["seat"]]
    alive_list = ", ".join([f"{p['name']} (Seat {p['seat']})" for p in alive_others])

    strategies = {
        "Civilian":  "Vote for whoever seems most suspicious — whose clue doesn't quite fit.",
        "Undercover": "Vote for a Civilian to help your team. Deflect suspicion.",
        "Mr. White": "Vote with the majority to blend in.",
    }

    return (
        f"You are {player['name']}. {pers['desc']}\n"
        f"Your role: {player['role']}. {strategies[player['role']]}\n\n"
        f"Previous rounds: {format_game_history_short(game_state)}\n"
        f"This round's clues:\n{format_current_round_clues(game_state)}\n\n"
        f"Surviving players to vote for: {alive_list}\n\n"
        "Vote to eliminate one player. Respond with ONLY:\n"
        "VOTE: <player name>\n"
    )


def build_bot_guess_prompt(player, game_state):
    """Prompt for an opponent bot Mr. White's guess."""
    return (
        f"You are {player['name']}, Mr. White, and you've been eliminated.\n"
        f"Game clues so far: {format_game_history_short(game_state)}\n"
        "Guess the Civilian word. Respond with ONLY:\nGUESS: <word>\n"
    )


# --------------------------------------------------------------------------------
# 7. GAME STATE FORMATTING HELPERS
# --------------------------------------------------------------------------------

MAX_HISTORY_CHARS = 10000


def format_game_history(game_state, viewer_seat):
    """Full game history from a specific player's perspective."""
    lines = []
    for rnd in game_state["completed_rounds"]:
        lines.append(f"\n--- Round {rnd['round_num']} ---")
        lines.append("Clues:")
        for seat, clue in rnd["clues"].items():
            p = game_state["players"][seat]
            lines.append(f"  {p['name']}: \"{clue}\"")
        lines.append(f"Votes: {format_votes(rnd['votes'], game_state)}")
        if rnd["eliminated"] is not None:
            elim_p = game_state["players"][rnd["eliminated"]]
            lines.append(f"Eliminated: {elim_p['name']} — revealed as {elim_p['role']}")
            if rnd.get("mr_white_guess"):
                correct = "CORRECT!" if rnd.get("mr_white_correct") else "WRONG"
                lines.append(f"  Mr. White guessed: \"{rnd['mr_white_guess']}\" — {correct}")
        else:
            lines.append("No one eliminated (tie).")

    full = "\n".join(lines)
    if len(full) > MAX_HISTORY_CHARS:
        while len("\n".join(lines)) > MAX_HISTORY_CHARS and len(lines) > 5:
            lines.pop(0)
        full = "... (earlier rounds truncated) ...\n" + "\n".join(lines)
    return full if full.strip() else "(No history yet — this is the first round.)"


def format_game_history_short(game_state):
    """Abbreviated history for bot prompts."""
    if not game_state["completed_rounds"]:
        return "(First round)"
    lines = []
    for rnd in game_state["completed_rounds"][-3:]:
        clue_strs = [f"{game_state['players'][s]['name']}:\"{c}\""
                     for s, c in rnd["clues"].items()]
        elim_str = ""
        if rnd["eliminated"] is not None:
            ep = game_state["players"][rnd["eliminated"]]
            elim_str = f" -> Eliminated {ep['name']}({ep['role']})"
        lines.append(f"R{rnd['round_num']}: {', '.join(clue_strs)}{elim_str}")
    return " | ".join(lines)


def format_current_round_clues(game_state):
    """Format clues given so far in the current round."""
    if not game_state["current_round_clues"]:
        return "(No clues yet this round.)"
    lines = []
    for seat, clue in game_state["current_round_clues"].items():
        p = game_state["players"][seat]
        lines.append(f"  {p['name']}: \"{clue}\"")
    return "\n".join(lines)


def format_votes(votes, game_state):
    """Format vote results."""
    if not votes:
        return "(no votes)"
    parts = []
    for voter_seat, target_seat in votes.items():
        v_name = game_state["players"][voter_seat]["name"]
        t_name = game_state["players"][target_seat]["name"]
        parts.append(f"{v_name}→{t_name}")
    return ", ".join(parts)


# --------------------------------------------------------------------------------
# 8. RESPONSE PARSERS
# --------------------------------------------------------------------------------

def parse_clue(raw):
    """Extract CLUE from response. Default: 'interesting'."""
    try:
        if "CLUE:" in raw:
            clue = raw.split("CLUE:")[1].strip().split("\n")[0].strip()
            clue = clue.strip('"').strip("'")
            words = clue.split()[:5]
            if words:
                return " ".join(words)
    except Exception:
        pass
    return "interesting"


def parse_vote(raw, alive_players, voter_seat):
    """Extract VOTE from response. Returns target seat or None."""
    try:
        if "VOTE:" in raw:
            vote_text = raw.split("VOTE:")[1].strip().split("\n")[0].strip().lower()
            for p in alive_players:
                if p["seat"] != voter_seat and p["name"].lower() in vote_text:
                    return p["seat"]
            for p in alive_players:
                if p["seat"] != voter_seat and str(p["seat"]) in vote_text:
                    return p["seat"]
    except Exception:
        pass
    return None


def parse_guess(raw):
    """Extract GUESS from Mr. White's response."""
    try:
        if "GUESS:" in raw:
            guess = raw.split("GUESS:")[1].strip().split("\n")[0].strip()
            return guess.strip('"').strip("'").lower()
    except Exception:
        pass
    return ""


# --------------------------------------------------------------------------------
# 9. GAME ENGINE
# --------------------------------------------------------------------------------

def determine_speaking_order(alive_seats, round_num, game_id):
    """Deterministic but varied speaking order per round."""
    rng = random.Random(game_id * 1000 + round_num * 13)
    order = list(alive_seats)
    rng.shuffle(order)
    return order


def tally_votes(votes, alive_seats):
    """Count votes, return eliminated seat. Ties: lowest seat wins."""
    if not votes:
        return None
    counts = {}
    for target in votes.values():
        counts[target] = counts.get(target, 0) + 1
    max_count = max(counts.values())
    candidates = [s for s, c in counts.items() if c == max_count]
    return min(candidates)


def check_win_condition(players, alive_seats):
    """Returns win condition string or None if game continues."""
    alive_roles = [players[s]["role"] for s in alive_seats]
    civ_count = alive_roles.count("Civilian")
    uc_count = alive_roles.count("Undercover")
    mw_count = alive_roles.count("Mr. White")

    if uc_count == 0 and mw_count == 0:
        return "civilian_win"
    if uc_count >= civ_count and uc_count > 0:
        return "undercover_win"
    if len(alive_seats) <= 2 and mw_count > 0:
        return "mr_white_survive_win"
    return None


def run_single_game(target_llm, opponent_llm, game_config):
    """
    Execute a single Undercover game.
    target_llm:   the model under test (plays model_seat)
    opponent_llm:  the fixed opponent model (plays all 5 other seats)
    Returns a result dict with all game data and scoring inputs.
    """
    game_id = game_config["game_id"]
    players, word_a, word_b = assign_roles(game_config)

    game_state = {
        "players": players,
        "alive": set(range(6)),
        "completed_rounds": [],
        "current_round_clues": {},
        "word_a": word_a,
        "word_b": word_b,
        "game_id": game_id,
    }

    model_seat = game_config["model_seat"]
    max_rounds = 5

    result = {
        "game_id": game_id,
        "model_role": game_config["model_role"],
        "perm_label": game_config["perm_label"],
        "word_a": word_a,
        "word_b": word_b,
        "difficulty": WORD_PAIRS[game_config["pair_index"]]["difficulty"],
        "purpose": game_config["purpose"],
        "model_seat": model_seat,
        "rounds_played": 0,
        "model_survived": True,
        "outcome": None,
        "model_clues": [],
        "model_votes": [],
        "model_vote_targets_roles": [],
        "mr_white_guess": None,
        "mr_white_guess_correct": False,
        "all_rounds_data": [],
    }

    win_condition = None

    for round_num in range(1, max_rounds + 1):
        if win_condition is not None:
            break

        alive_seats = sorted(game_state["alive"])
        if len(alive_seats) <= 1:
            break

        result["rounds_played"] = round_num
        game_state["current_round_clues"] = {}

        # ── CLUE PHASE ──
        speaking_order = determine_speaking_order(alive_seats, round_num, game_id)

        for seat in speaking_order:
            player = players[seat]
            if player["is_model"]:
                # ─── TARGET LLM gives clue ───
                prompt = build_model_clue_prompt(player, game_state, round_num)
                with kbench.chats.new(f"g{game_id}_r{round_num}_target_clue"):
                    raw = target_llm.prompt(prompt)
                clue = parse_clue(raw)
                result["model_clues"].append(clue)
            else:
                # ─── OPPONENT LLM gives clue ───
                prompt = build_bot_clue_prompt(player, game_state, round_num)
                with kbench.chats.new(f"g{game_id}_r{round_num}_opp{seat}_clue"):
                    raw = opponent_llm.prompt(prompt)
                clue = parse_clue(raw)

            game_state["current_round_clues"][seat] = clue

        # ── VOTE PHASE ──
        votes = {}
        alive_players = [players[s] for s in alive_seats]

        for seat in alive_seats:
            player = players[seat]
            if player["is_model"]:
                # ─── TARGET LLM votes ───
                prompt = build_model_vote_prompt(player, game_state, round_num)
                with kbench.chats.new(f"g{game_id}_r{round_num}_target_vote"):
                    raw = target_llm.prompt(prompt)
                target = parse_vote(raw, alive_players, seat)
                if target is None:
                    others = [s for s in alive_seats if s != seat]
                    rng = random.Random(game_id * 100 + round_num * 10 + seat)
                    target = rng.choice(others)
                result["model_votes"].append(target)
                result["model_vote_targets_roles"].append(players[target]["role"])
            else:
                # ─── OPPONENT LLM votes ───
                prompt = build_bot_vote_prompt(player, game_state, round_num)
                with kbench.chats.new(f"g{game_id}_r{round_num}_opp{seat}_vote"):
                    raw = opponent_llm.prompt(prompt)
                target = parse_vote(raw, alive_players, seat)
                if target is None:
                    others = [s for s in alive_seats if s != seat]
                    rng = random.Random(game_id * 100 + round_num * 10 + seat + 99)
                    target = rng.choice(others)

            votes[seat] = target

        # ── ELIMINATION ──
        eliminated_seat = tally_votes(votes, alive_seats)
        mr_white_guess = None
        mr_white_correct = False

        if eliminated_seat is not None:
            elim_player = players[eliminated_seat]
            game_state["alive"].discard(eliminated_seat)

            if eliminated_seat == model_seat:
                result["model_survived"] = False

            # Mr. White gets a guess on elimination
            if elim_player["role"] == "Mr. White":
                if elim_player["is_model"]:
                    # ─── TARGET LLM guesses as Mr. White ───
                    prompt = build_model_guess_prompt(elim_player, game_state)
                    with kbench.chats.new(f"g{game_id}_r{round_num}_target_mw_guess"):
                        raw = target_llm.prompt(prompt)
                    guess = parse_guess(raw)
                    mr_white_correct = (guess == word_a.lower())
                    result["mr_white_guess"] = guess
                    result["mr_white_guess_correct"] = mr_white_correct
                else:
                    # ─── OPPONENT LLM guesses as Mr. White ───
                    prompt = build_bot_guess_prompt(elim_player, game_state)
                    with kbench.chats.new(f"g{game_id}_r{round_num}_opp_mw_guess"):
                        raw = opponent_llm.prompt(prompt)
                    guess = parse_guess(raw)
                    mr_white_correct = (guess == word_a.lower())
                    mr_white_guess = guess

                if mr_white_correct:
                    win_condition = "mr_white_guess_win"

        round_data = {
            "round_num": round_num,
            "clues": dict(game_state["current_round_clues"]),
            "votes": dict(votes),
            "eliminated": eliminated_seat,
            "mr_white_guess": mr_white_guess or result.get("mr_white_guess"),
            "mr_white_correct": mr_white_correct,
        }
        game_state["completed_rounds"].append(round_data)
        result["all_rounds_data"].append(round_data)

        if win_condition is None:
            win_condition = check_win_condition(players, sorted(game_state["alive"]))

    if win_condition is None:
        win_condition = "timeout"

    result["outcome"] = win_condition
    result["final_alive"] = sorted(game_state["alive"])
    result["game_state"] = game_state

    return result


# --------------------------------------------------------------------------------
# 10. SCORING ENGINE
# --------------------------------------------------------------------------------

def _get_survived_rounds(result):
    """Calculate how many rounds the model survived before elimination."""
    if result["model_survived"]:
        return result["rounds_played"]
    model_seat = result["model_seat"]
    for rd in result["all_rounds_data"]:
        if rd["eliminated"] == model_seat:
            return max(rd["round_num"] - 1, 0)
    return result["rounds_played"]


def score_civilian_game(result):
    """Score a game where the model played Civilian. Returns 0.0-1.0."""
    score = 0.0

    # Base: +0.60 for civilian team win
    if result["outcome"] == "civilian_win":
        score += 0.60

    # Survival: +0.15 if model survived
    if result["model_survived"]:
        score += 0.15

    # Vote accuracy: +0.15 × fraction targeting non-civilians
    if result["model_vote_targets_roles"]:
        correct = sum(1 for r in result["model_vote_targets_roles"]
                      if r in ("Undercover", "Mr. White"))
        score += 0.15 * (correct / len(result["model_vote_targets_roles"]))

    # Clue quality via judge: +0.10
    if result["model_clues"]:
        clue_str = ", ".join([f'"{c}"' for c in result["model_clues"]])
        try:
            assessment = kbench.assertions.assess_response_with_judge(
                response_text=(
                    f"Word: \"{result['word_a']}\". Clues given: {clue_str}. "
                    f"Similar undercover word: \"{result['word_b']}\"."
                ),
                judge_llm=kbench.judge_llm,
                criteria=[
                    "The clues should describe the given word accurately and specifically.",
                    "The clues should be subtle enough to be distinguishable from the similar undercover word.",
                ],
            )
            passed = sum(1 for r in assessment.results if r.passed)
            score += 0.10 * (passed / max(len(assessment.results), 1))
        except Exception:
            pass

    return min(score, 1.0)


def score_undercover_game(result):
    """Score a game where the model played Undercover. Returns 0.0-1.0."""
    score = 0.0

    # Base
    if result["outcome"] == "undercover_win":
        score += 0.60
    elif result["model_survived"] and result["rounds_played"] >= 3:
        score += 0.30

    # Survival depth: +0.15
    max_rounds = result["rounds_played"]
    if max_rounds > 0:
        survived = _get_survived_rounds(result)
        score += 0.15 * (survived / max(max_rounds, 1))

    # Deception consistency via judge: +0.15
    if result["model_clues"]:
        clue_str = ", ".join([f'"{c}"' for c in result["model_clues"]])
        try:
            assessment = kbench.assertions.assess_response_with_judge(
                response_text=(
                    f"Undercover word: \"{result['word_b']}\". Civilian word: \"{result['word_a']}\". "
                    f"Undercover player's clues: {clue_str}."
                ),
                judge_llm=kbench.judge_llm,
                criteria=[
                    "The clues should be plausible descriptions of the CIVILIAN word while actually fitting the undercover word — effective social camouflage.",
                    "The clues should be consistent with each other across rounds, not contradictory.",
                ],
            )
            passed = sum(1 for r in assessment.results if r.passed)
            score += 0.15 * (passed / max(len(assessment.results), 1))
        except Exception:
            pass

    # Vote misdirection: +0.10
    if result["model_vote_targets_roles"]:
        misdirect = sum(1 for r in result["model_vote_targets_roles"] if r == "Civilian")
        score += 0.10 * (misdirect / len(result["model_vote_targets_roles"]))

    return min(score, 1.0)


def score_mr_white_game(result):
    """Score a game where the model played Mr. White. Returns 0.0-1.0."""
    score = 0.0

    # Base
    if result["mr_white_guess_correct"]:
        score += 0.60
    elif result["outcome"] == "mr_white_survive_win":
        score += 0.50
    elif result["model_survived"]:
        score += 0.30
    elif result["mr_white_guess"] and not result["mr_white_guess_correct"]:
        score += 0.10

    # Survival depth: +0.15
    max_rounds = result["rounds_played"]
    if max_rounds > 0:
        survived = _get_survived_rounds(result)
        score += 0.15 * (survived / max(max_rounds, 1))

    # Clue plausibility via judge: +0.15
    if result["model_clues"]:
        clue_str = ", ".join([f'"{c}"' for c in result["model_clues"]])
        try:
            assessment = kbench.assertions.assess_response_with_judge(
                response_text=(
                    f"Mr. White (no word) gave these clues: {clue_str}. "
                    f"The civilian word was \"{result['word_a']}\" and undercover word was \"{result['word_b']}\"."
                ),
                judge_llm=kbench.judge_llm,
                criteria=[
                    "The clues should be socially plausible — sounding like they could describe a real word.",
                    "The clues should show adaptation to others' clues, not random guessing.",
                ],
            )
            passed = sum(1 for r in assessment.results if r.passed)
            score += 0.15 * (passed / max(len(assessment.results), 1))
        except Exception:
            pass

    # Inference quality: +0.10
    if result["mr_white_guess"]:
        if result["mr_white_guess_correct"]:
            score += 0.10
        elif result["mr_white_guess"] == result["word_b"].lower():
            score += 0.05  # Close — guessed the undercover word

    return min(score, 1.0)


def score_game(result):
    """Route to role-specific scorer."""
    role = result["model_role"]
    if role == "Civilian":
        return score_civilian_game(result)
    elif role == "Undercover":
        return score_undercover_game(result)
    elif role == "Mr. White":
        return score_mr_white_game(result)
    return 0.0


# --------------------------------------------------------------------------------
# 11. UI RENDERER
# --------------------------------------------------------------------------------

def render_game_state(result, game_state, round_data, round_num, status="playing"):
    """Render a single round's state as HTML."""
    players = game_state["players"]
    alive = game_state["alive"]

    status_bg = "#0d1117"
    if status == "won":
        status_bg = "#0a1f0a"
    elif status == "lost":
        status_bg = "#1f0a0a"

    player_html = ""
    for p in players:
        seat = p["seat"]
        is_alive = seat in alive
        is_model = p["is_model"]
        border_color = "#58a6ff" if is_model else "#30363d"
        bg = "#161b22" if is_alive else "#0d1117"
        opacity = "1" if is_alive else "0.4"

        role_badge = ""
        if not is_alive:
            role_colors = {"Civilian": "#3fb950", "Undercover": "#f85149", "Mr. White": "#d29922"}
            role_badge = (
                f'<span style="font-size:9px;color:{role_colors.get(p["role"], "#8b949e")};">'
                f'{p["role"]}</span>'
            )

        clue = round_data["clues"].get(seat, "") if round_data else ""
        vote_target = round_data["votes"].get(seat) if round_data else None
        vote_name = players[vote_target]["name"] if vote_target is not None else ""

        elim_marker = ""
        if round_data and round_data["eliminated"] == seat:
            elim_marker = ' <span style="color:#f85149;">✕</span>'

        model_marker = ' <span style="color:#58a6ff;font-size:9px;">★ YOU</span>' if is_model else ""

        player_html += (
            f'<div style="background:{bg};border:1px solid {border_color};border-radius:6px;'
            f'padding:8px;opacity:{opacity};min-width:90px;">'
            f'<div style="font-weight:600;font-size:12px;color:#f0f6fc;">'
            f'{p["name"]}{model_marker}{elim_marker}</div>'
            f'{role_badge}'
            f'<div style="font-size:11px;color:#c9d1d9;margin-top:4px;">'
            f'Clue: <i>{clue if clue else "—"}</i></div>'
            f'<div style="font-size:10px;color:#8b949e;">Vote: {vote_name if vote_name else "—"}</div>'
            f'</div>'
        )

    elim_text = ""
    if round_data and round_data["eliminated"] is not None:
        ep = players[round_data["eliminated"]]
        elim_text = (
            f'Eliminated: <b>{ep["name"]}</b> — revealed as '
            f'<span style="color:#f85149;">{ep["role"]}</span>'
        )
        if round_data.get("mr_white_guess"):
            g = round_data["mr_white_guess"]
            c = "color:#3fb950;" if round_data.get("mr_white_correct") else "color:#f85149;"
            elim_text += f'<br/>Mr. White guessed: <span style="{c}">"{g}"</span>'

    diff = result["difficulty"]
    pair_label = f"{result['word_a']} / {result['word_b']}"

    return f"""
    <div style="background:{status_bg};padding:14px;border-radius:10px;border:1px solid #30363d;
                font-family:'Segoe UI',system-ui,sans-serif;color:#c9d1d9;margin:4px 0;max-width:760px;">
        <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:10px;">
            <span style="font-weight:600;font-size:14px;color:#f0f6fc;">
                G{result['game_id']}{result['perm_label']}: {result['model_role']} · {pair_label} ({diff})</span>
            <span style="font-size:12px;color:#8b949e;">Round {round_num}</span>
        </div>
        <div style="display:flex;gap:6px;flex-wrap:wrap;margin-bottom:10px;">
            {player_html}
        </div>
        <div style="font-size:11px;color:#c9d1d9;padding:6px 8px;background:#21262d;border-radius:4px;">
            {elim_text if elim_text else "No elimination this round."}
        </div>
    </div>"""


def render_game_summary(result, score):
    """Render end-of-game summary."""
    outcome_labels = {
        "civilian_win": ("Civilians Win!", "#3fb950"),
        "undercover_win": ("Undercover Wins!", "#f85149"),
        "mr_white_guess_win": ("Mr. White Wins! (Correct Guess)", "#d29922"),
        "mr_white_survive_win": ("Mr. White Survives!", "#d29922"),
        "timeout": ("Timeout — No Winner", "#8b949e"),
    }
    label, color = outcome_labels.get(result["outcome"], ("Unknown", "#8b949e"))

    survived_icon = "✅" if result["model_survived"] else "❌"
    clue_list = ", ".join([f'"{c}"' for c in result["model_clues"]]) if result["model_clues"] else "—"

    guess_line = ""
    if result["model_role"] == "Mr. White" and result["mr_white_guess"]:
        g_icon = "✅" if result["mr_white_guess_correct"] else "❌"
        guess_line = (
            f'<div style="font-size:11px;margin-top:4px;">'
            f'Mr. White Guess: {g_icon} "{result["mr_white_guess"]}"</div>'
        )

    return f"""
    <div style="background:#0d1117;padding:14px;border-radius:10px;border:1px solid {color};
                font-family:'Segoe UI',system-ui,sans-serif;color:#c9d1d9;margin:6px 0;max-width:760px;">
        <div style="display:flex;justify-content:space-between;align-items:center;">
            <span style="font-weight:700;font-size:14px;color:{color};">
                G{result['game_id']}{result['perm_label']}: {label}</span>
            <span style="font-size:20px;font-weight:700;color:#58a6ff;">{score:.3f}</span>
        </div>
        <div style="font-size:11px;color:#8b949e;margin-top:6px;">
            Role: {result['model_role']} · Words: "{result['word_a']}" vs "{result['word_b']}"
            ({result['difficulty']}) · Rounds: {result['rounds_played']} · Survived: {survived_icon}
        </div>
        <div style="font-size:11px;color:#8b949e;margin-top:4px;">Model clues: {clue_list}</div>
        {guess_line}
    </div>"""


def render_final_scoreboard(game_scores, final_score):
    """Render the final benchmark scoreboard."""
    rows = ""
    for gs in game_scores:
        color = "#3fb950" if gs["score"] >= 0.5 else "#f85149" if gs["score"] < 0.2 else "#d29922"
        rows += (
            f'<div style="display:flex;justify-content:space-between;padding:4px 0;'
            f'border-bottom:1px solid #21262d;font-size:12px;">'
            f'<span>G{gs["game_id"]}{gs["perm"]}: {gs["role"]} · {gs["words"]} ({gs["difficulty"]})</span>'
            f'<span style="color:{color};font-weight:600;">{gs["score"]:.3f}</span>'
            f'</div>'
        )

    # Role averages
    role_groups = {}
    for gs in game_scores:
        role_groups.setdefault(gs["role"], []).append(gs["score"])
    role_avgs = ""
    for role in ["Civilian", "Undercover", "Mr. White"]:
        if role in role_groups:
            avg = sum(role_groups[role]) / len(role_groups[role])
            role_avgs += f'<span style="margin-right:16px;">{role}: {avg:.3f}</span>'

    # Difficulty averages
    diff_groups = {}
    for gs in game_scores:
        diff_groups.setdefault(gs["difficulty"], []).append(gs["score"])
    diff_avgs = ""
    for diff in ["easy", "medium", "hard"]:
        if diff in diff_groups:
            avg = sum(diff_groups[diff]) / len(diff_groups[diff])
            diff_avgs += f'<span style="margin-right:16px;">{diff}: {avg:.3f}</span>'

    solved = sum(1 for gs in game_scores if gs["score"] >= 0.5)
    total = len(game_scores)

    return f"""
    <div style="background:#0d1117;padding:20px;border-radius:10px;border:1px solid #58a6ff;
                font-family:'Segoe UI',system-ui,sans-serif;color:#f0f6fc;text-align:center;
                margin:12px 0;max-width:760px;">
        <div style="font-size:13px;color:#8b949e;margin-bottom:4px;">
            MULTI-PARTY SOCIAL NAVIGATION: UNDERCOVER</div>
        <div style="font-size:36px;font-weight:700;color:#58a6ff;">{final_score:.4f}</div>
        <div style="font-size:13px;color:#8b949e;margin-top:4px;">
            Final Score (0.0 – 1.0) · {solved}/{total} games won</div>
        <div style="font-size:11px;color:#8b949e;margin-top:4px;">
            Opponent: {OPPONENT_MODEL_ID}</div>
        <div style="font-size:11px;color:#8b949e;margin-top:8px;">
            By Role: {role_avgs}</div>
        <div style="font-size:11px;color:#8b949e;margin-top:4px;">
            By Difficulty: {diff_avgs}</div>
        <div style="text-align:left;margin-top:12px;padding-top:8px;border-top:1px solid #30363d;">
            {rows}
        </div>
    </div>"""


# --------------------------------------------------------------------------------
# 12. THE BENCHMARK TASK
# --------------------------------------------------------------------------------

@kbench.task(name="undercover_social_navigation")
def undercover_benchmark(llm) -> float:
    """
    Main benchmark entry point.
    `llm` = the target model under test (variable, swapped by Kaggle harness).
    Opponent bots ALWAYS use the hardcoded OPPONENT_MODEL_ID.

    27 games total: 9 word pairs × 3 role permutations.
    """
    # ── Resolve the fixed opponent model ──
    opponent_llm = kbench.llms[OPPONENT_MODEL_ID]

    all_game_scores = []

    for config in GAME_CONFIGS:
        # Run the game: target llm vs fixed opponent llm
        result = run_single_game(llm, opponent_llm, config)

        # Display each round
        game_state = result["game_state"]
        for rd in result["all_rounds_data"]:
            is_last = (rd == result["all_rounds_data"][-1])
            status = "playing"
            if is_last:
                role = result["model_role"]
                outcome = result["outcome"]
                if role == "Civilian" and outcome == "civilian_win":
                    status = "won"
                elif role == "Undercover" and outcome == "undercover_win":
                    status = "won"
                elif role == "Mr. White" and outcome in ("mr_white_guess_win", "mr_white_survive_win"):
                    status = "won"
                else:
                    status = "lost"
            display(HTML(render_game_state(result, game_state, rd, rd["round_num"], status)))

        # Score the game
        game_score = score_game(result)

        all_game_scores.append({
            "game_id": config["game_id"],
            "perm": config["perm_label"],
            "role": config["model_role"],
            "difficulty": WORD_PAIRS[config["pair_index"]]["difficulty"],
            "words": f"{result['word_a']} vs {result['word_b']}",
            "score": game_score,
        })

        display(HTML(render_game_summary(result, game_score)))

    # ── Final aggregate score ──
    final_score = sum(gs["score"] for gs in all_game_scores) / len(all_game_scores)
    display(HTML(render_final_scoreboard(all_game_scores, final_score)))

    return final_score


# --------------------------------------------------------------------------------
# 13. EXECUTION
# --------------------------------------------------------------------------------
undercover_benchmark.run(kbench.llm)